In [2]:
import json
import operator
import os
from pathlib import Path
from typing import Annotated, Literal

from dotenv import load_dotenv
from openai import OpenAI
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


In [3]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError("Can not found the project root")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")

client = (
    OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url="https://api.deepseek.com",
        timeout=60.0,
        max_retries=2,
    )
    if DEEPSEEK_API_KEY
    else None
)

if client is None:
    print("DEEPSEEK_API_KEY is empty")
else:
    print(f"DeepSeek client ready — model: {MODEL}")


DeepSeek client ready — model: deepseek-v4-flash


## LLM helper

Thinking mode được tắt cho lab này để classifier/evaluator nhanh và dễ quan sát. JSON mode chỉ được bật ở các node cần structured output.


In [4]:
def call_llm(
    system_prompt: str,
    user_prompt: str,
    *,
    json_mode: bool = False,
    max_tokens: int = 500,
) -> str:
    if client is None:
        raise RuntimeError(
            "DeepSeek chưa được cấu hình. Điền DEEPSEEK_API_KEY trong .env "
            "và chạy lại cell cấu hình."
        )

    kwargs = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "extra_body": {"thinking": {"type": "disabled"}},
        "max_tokens": max_tokens,
    }

    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    response = client.chat.completions.create(**kwargs)
    content = response.choices[0].message.content

    if not content:
        raise RuntimeError("DeepSeek trả về nội dung rỗng")

    return content


In [5]:
if client is None:
    print("Bỏ qua smoke test vì chưa có API key.")
else:
    response = call_llm(
        system_prompt="You are a helpful AI assistant.",
        user_prompt="Explain Graph Engineering in one sentence.",
        max_tokens=100,
    )
    print(response)


Graph engineering is the discipline of designing, building, and optimizing graph-based data structures and algorithms—such as social networks, knowledge graphs, or recommendation systems—to efficiently model, store, and query complex relationships at scale.


## State

In [6]:
class GraphState(TypedDict, total=False):
    query: str
    category: str
    evidence: list[str]
    analysis: str
    score: float
    feedback: str
    attempts: int
    final_answer: str
    route_log: Annotated[list[str], operator.add]


## Classifier node — LLM

In [7]:
def classify(state: GraphState):
    prompt = f"""
Classify the user's query into exactly one category.

Available categories:
- weather: climate, temperature, rainfall, heatwaves, cooling demand
- trade: import, export, tariff, customs, international trade
- politics: government policy, regulation, geopolitical or political events

User query:
{state["query"]}

Return only JSON in this format:
{{"category": "weather"}}

The category must be exactly one of: weather, trade, politics.
"""

    raw = call_llm(
        system_prompt=(
            "You are a routing classifier inside an AI workflow. "
            "Always return valid JSON."
        ),
        user_prompt=prompt,
        json_mode=True,
        max_tokens=100,
    )
    data = json.loads(raw)
    category = str(data.get("category", "")).lower().strip()
    allowed = {"weather", "trade", "politics"}

    if category not in allowed:
        raise ValueError(f"Invalid category from LLM: {category!r}")

    return {
        "category": category,
        "route_log": [f"classify_llm -> {category}"],
    }


## Research nodes — Python



In [8]:
def weather_research(state: GraphState):
    return {
        "evidence": [
            "High temperatures generally increase cooling demand.",
            "Long heatwaves can increase peak electricity consumption.",
        ],
        "route_log": ["weather_research"],
    }


def trade_research(state: GraphState):
    return {
        "evidence": [
            "Import volumes can indicate changes in market activity.",
            "Tariff changes can affect product competitiveness.",
        ],
        "route_log": ["trade_research"],
    }


def politics_research(state: GraphState):
    return {
        "evidence": [
            "Government policy can influence business conditions.",
            "Geopolitical events can affect supply-chain risk.",
        ],
        "route_log": ["politics_research"],
    }


## Analyst node — LLM

In [9]:
def analyze(state: GraphState):
    evidence_text = "\n".join(
        f"- {item}" for item in state.get("evidence", [])
    )

    prompt = f"""
Analyze the information below.

User query:
{state["query"]}

Category:
{state["category"]}

Evidence:
{evidence_text}

Evaluator feedback from the previous iteration:
{state.get("feedback", "None")}

Requirements:
1. Answer the user's question.
2. Use only the supplied evidence.
3. Explain the causal relationship.
4. Do not invent additional facts.
5. Keep the analysis concise.

Return plain text.
"""

    analysis = call_llm(
        system_prompt="You are a market intelligence analyst.",
        user_prompt=prompt,
        max_tokens=500,
    )

    return {
        "analysis": analysis,
        "route_log": ["analyze_llm"],
    }


## Evaluator node — LLM

In [10]:
def evaluate(state: GraphState):
    evidence_json = json.dumps(
        state.get("evidence", []),
        ensure_ascii=False,
        indent=2,
    )

    prompt = f"""
Evaluate the following analysis.

Original query:
{state["query"]}

Evidence:
{evidence_json}

Analysis:
{state["analysis"]}

Evaluate whether the analysis answers the query, is supported by evidence,
has clear causal reasoning, and has enough evidence.

Return JSON exactly in this format:
{{
  "score": 0.0,
  "feedback": "..."
}}

The score must be between 0.0 and 1.0. There are initially only two evidence
items, so score conservatively when the evidence is limited.
"""

    raw = call_llm(
        system_prompt=(
            "You are a strict evaluator inside an AI workflow. "
            "Always return valid JSON."
        ),
        user_prompt=prompt,
        json_mode=True,
        max_tokens=200,
    )
    result = json.loads(raw)
    score = max(0.0, min(1.0, float(result["score"])))
    feedback = str(result.get("feedback", "No feedback provided.")).strip()

    return {
        "score": score,
        "feedback": feedback,
        "route_log": [f"evaluate_llm -> {score:.2f}"],
    }


## Improvement node — Python

In [11]:
def improve(state: GraphState):
    evidence = list(state.get("evidence", []))
    attempts = state.get("attempts", 0) + 1

    evidence.append(
        "Additional evidence added after "
        f"evaluation attempt {attempts}: multiple independent indicators "
        "should be considered before drawing a strong conclusion."
    )

    return {
        "evidence": evidence,
        "attempts": attempts,
        "route_log": [f"improve -> attempt={attempts}"],
    }


## Finalizer node — LLM

In [12]:
def finalize(state: GraphState):
    prompt = f"""
Create the final answer for the user.

Question:
{state["query"]}

Analysis:
{state["analysis"]}

Evaluation score:
{state["score"]}

Evaluator feedback:
{state["feedback"]}

Requirements: concise, clear, evidence-grounded, and no unsupported claims.
"""

    answer = call_llm(
        system_prompt="You are the final response writer.",
        user_prompt=prompt,
        max_tokens=400,
    )

    return {
        "final_answer": answer,
        "route_log": ["finalize_llm"],
    }


## Routers — deterministic control

In [13]:
def route_category(
    state: GraphState,
) -> Literal["weather", "trade", "politics"]:
    return state["category"]


def route_after_evaluation(
    state: GraphState,
) -> Literal["retry", "finish"]:
    if state["score"] >= 0.8:
        return "finish"

    if state.get("attempts", 0) >= 3:
        return "finish"

    return "retry"


## Build and compile the graph

In [14]:
builder = StateGraph(GraphState)

builder.add_node("classify", classify)
builder.add_node("weather_research", weather_research)
builder.add_node("trade_research", trade_research)
builder.add_node("politics_research", politics_research)
builder.add_node("analyze", analyze)
builder.add_node("evaluate", evaluate)
builder.add_node("improve", improve)
builder.add_node("finalize", finalize)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_category,
    {
        "weather": "weather_research",
        "trade": "trade_research",
        "politics": "politics_research",
    },
)
builder.add_edge("weather_research", "analyze")
builder.add_edge("trade_research", "analyze")
builder.add_edge("politics_research", "analyze")
builder.add_edge("analyze", "evaluate")
builder.add_conditional_edges(
    "evaluate",
    route_after_evaluation,
    {
        "retry": "improve",
        "finish": "finalize",
    },
)
builder.add_edge("improve", "analyze")
builder.add_edge("finalize", END)

graph = builder.compile()


## Visualize

In [15]:
print(graph.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	weather_research(weather_research)
	trade_research(trade_research)
	politics_research(politics_research)
	analyze(analyze)
	evaluate(evaluate)
	improve(improve)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	analyze --> evaluate;
	classify -. &nbsp;politics&nbsp; .-> politics_research;
	classify -. &nbsp;trade&nbsp; .-> trade_research;
	classify -. &nbsp;weather&nbsp; .-> weather_research;
	evaluate -. &nbsp;finish&nbsp; .-> finalize;
	evaluate -. &nbsp;retry&nbsp; .-> improve;
	improve --> analyze;
	politics_research --> analyze;
	trade_research --> analyze;
	weather_research --> analyze;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Run the graph

In [ ]:
initial_state = {
    "query": "How can heatwaves affect air-conditioner demand?",
    "attempts": 0,
    "route_log": [],
}

if client is None:
    result = None
    print("API key is empty")
else:
    result = graph.invoke(initial_state, {"recursion_limit": 20})
    print(result["final_answer"])


Heatwaves clearly increase air-conditioner demand: sustained high temperatures directly raise the need for cooling, causing more people to run AC units. During a prolonged heatwave, this also boosts peak electricity consumption, as many households and businesses cool their spaces at the same time.

However, the available evidence only supports the direction of this effect—that heatwaves raise AC demand—not its precise magnitude. The evidence does not specify how large the increase is, nor does it account for moderating factors such as building efficiency, regional climate, or use of alternative cooling methods. Therefore, while the causal link is well-founded, any quantification of the demand increase would require additional data and would be unsupported by the current evidence.


## Inspect the execution path

In [17]:
if result is None:
    print("Chưa có execution trace. Hãy cấu hình API key và chạy graph trước.")
else:
    print("EXECUTION TRACE")
    print("=" * 50)

    for index, step in enumerate(result["route_log"], start=1):
        print(f"{index:02d}. {step}")


EXECUTION TRACE
01. classify_llm -> weather
02. weather_research
03. analyze_llm
04. evaluate_llm -> 0.60
05. improve -> attempt=1
06. analyze_llm
07. evaluate_llm -> 0.70
08. improve -> attempt=2
09. analyze_llm
10. evaluate_llm -> 0.70
11. improve -> attempt=3
12. analyze_llm
13. evaluate_llm -> 0.60
14. finalize_llm


## Stream node updates 

To see how nodes update


In [18]:
if client is None:
    print("Chưa stream graph vì chưa có API key.")
else:
    for event in graph.stream(
        initial_state,
        {"recursion_limit": 20},
        stream_mode="updates",
    ):
        print(event)
        print("-" * 80)


{'classify': {'category': 'weather', 'route_log': ['classify_llm -> weather']}}
--------------------------------------------------------------------------------
{'weather_research': {'evidence': ['High temperatures generally increase cooling demand.', 'Long heatwaves can increase peak electricity consumption.'], 'route_log': ['weather_research']}}
--------------------------------------------------------------------------------
{'analyze': {'analysis': 'Heatwaves increase air-conditioner demand because high temperatures directly raise the need for cooling. This higher demand for cooling, sustained over a long heatwave, leads to a rise in peak electricity consumption, as more air-conditioners run simultaneously and for longer periods.', 'route_log': ['analyze_llm']}}
--------------------------------------------------------------------------------
{'evaluate': {'score': 0.7, 'feedback': 'The analysis directly addresses the query by explaining how heatwaves increase air-conditioner demand 